## Desafio D — Sistema de Registro de Preços

### Problema

Quais características diferenciam contratações realizadas com e sem Sistema de Registro de Preços?

### Possíveis perguntas

- O SRP é mais frequente em determinados tipos de contratação?
- Existem diferenças nos valores das contratações?
- Determinados órgãos utilizam SRP proporcionalmente mais do que outros?

### Variável de interesse

Quando disponível:

```text
srp
```

### Possíveis análises

- proporções;
- tabelas cruzadas;
- comparação de valores;
- teste qui-quadrado.


documentação API: https://dadosabertos.compras.gov.br/swagger-ui/index.html


In [1]:
#!pip install requests pandas matplotlib -q
#!pip install pyarrow

In [2]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import time

In [3]:
BASE_URL = "https://dadosabertos.compras.gov.br"

ENDPOINT_CONTRATACOES = "/modulo-contratacoes/1_consultarContratacoes_PNCP_14133"

url = BASE_URL + ENDPOINT_CONTRATACOES

print(url)

https://dadosabertos.compras.gov.br/modulo-contratacoes/1_consultarContratacoes_PNCP_14133


In [4]:
def extrair_registros(json_resposta):
    if isinstance(json_resposta, list):
        return json_resposta

    if not isinstance(json_resposta, dict):
        return []

    for chave in ["resultado", "resultados", "data", "content"]:
        if chave in json_resposta and isinstance(json_resposta[chave], list):
            return json_resposta[chave]

    return []



# Pegando dados 2024

In [5]:
modalidades = [5, 6] # Pegar todas as modalidades de licitação, mas para fins de teste, vamos pegar apenas as principais.
todos_registros = []

In [ ]:
for modalidade in modalidades:
    pagina = 1
    total_modalidade = 0

    while True:
        params = {
            "pagina": pagina,
            "tamanhoPagina": 500,
            "dataPublicacaoPncpInicial": "2024-01-01",
            "dataPublicacaoPncpFinal": "2024-12-31",
            "codigoModalidade": modalidade,
            "unidadeOrgaoUfSigla": "SP"
        }

        resposta = requests.get(url, params=params, timeout=60)

        if resposta.status_code == 429:
            espera = 5
            print(f"Modalidade {modalidade}, página {pagina}: 429, aguardando {espera}s e tentando de novo...")
            time.sleep(espera)
            continue  # tenta a mesma página de novo, sem avançar

        if resposta.status_code != 200:
            print(f"Modalidade {modalidade}, página {pagina}: erro {resposta.status_code}")
            break

        dados = resposta.json()
        registros = extrair_registros(dados)

        if not registros:
            break

        todos_registros.extend(registros)
        total_modalidade += len(registros)

        if len(registros) < 500:
            break

        pagina += 1
        time.sleep(0.2)  # pausa maior entre páginas

    print(f"Modalidade {modalidade}: {total_modalidade} registros coletados.")
    time.sleep(3)  # pausa entre modalidades, para o servidor "esfriar"

Modalidade 5: 33327 registros coletados.


In [ ]:
# print("JSON retornado:", registros)
df_24 = pd.json_normalize(todos_registros)
df_24.head()

In [ ]:
#for i in todos_registros:
    #print(i)

In [ ]:

# 2. PROCURA AUTOMÁTICA: Procura qualquer coluna que tenha 'srp' ou 'preco' no nome
colunas_srp = [
    c
    for c in df_24.columns
    if 'srp' in c.lower() or 'registropreco' in c.lower() or 'preco' in c.lower()
]
print("Colunas encontradas relacionadas a SRP/Preço:")
print(colunas_srp)

# Se encontrou alguma coluna compatível, renomeia a primeira para 'srp'
if colunas_srp:
    coluna_identificada = colunas_srp[0]
    print(f"\nUsando a coluna '{coluna_identificada}' como 'srp'")
    df_24 = df_24.rename(columns={coluna_identificada: 'srp'})
else:
    print(
        "\nNenhuma coluna de SRP foi encontrada diretamente. Veja todas as colunas:"
    )
    print(df_24.columns.tolist())

# 3. Mapeia outras colunas comuns do PNCP
mapeamento = {
    'orgaoEntidade.razaoSocial': 'orgaoEntidadeRazaoSocial',
    'modalidadeNome': 'modalidadeNome',
    'numeroCompra': 'numeroCompra',
    'objetoCompra': 'objetoCompra',
    'valorTotalEstimado': 'valorTotalEstimado',
    'valorTotalHomologado': 'valorTotalHomologado',
}
df_24 = df_24.rename(
    columns={k: v for k, v in mapeamento.items() if k in df_24.columns}
)

df_24["ano"] = 2024


# 4. Trata e padroniza a coluna SRP (identifica True, 1, 'True', 'S', etc.)
if 'srp' in df_24.columns:
    # Mostra os valores brutos que vieram da API antes de converter
    print("\nValores brutos encontrados na coluna SRP:")
    print(df_24['srp'].value_counts(dropna=False))

    # Converte para booleano real
    df_24['srp_bool'] = df_24['srp'].astype(str).str.lower().isin(['true', '1', 's', 'sim'])

    df_24_com_srp = df_24[df_24['srp_bool'] == True]
    df_24_sem_srp = df_24[df_24['srp_bool'] == False]

    print(f"\n Total COM SRP: {len(df_24_com_srp)}")
    print(f" Total SEM SRP: {len(df_24_sem_srp)}")


In [ ]:
print("\nDataFrame com SRP:")
display(df_24_com_srp.head())
print("\nDataFrame sem SRP:")
display(df_24_sem_srp.head())

In [ ]:
# GroupBy agregando com nunique (únicos) e count (total de registros)
df_qnt_modalidade_orgao_24 = df_24.groupby('srp').agg(
    qtd_modalidades=('modalidadeNome', 'nunique'),
    qtd_orgaos=('orgaoEntidadeRazaoSocial', 'nunique'),
    total_registros=('numeroCompra', 'count')
)

# Renomeia os índices para facilitar a leitura no relatório
df_qnt_modalidade_orgao_24.index = ['Sem SRP', 'Com SRP']
display(df_qnt_modalidade_orgao_24)

# Pegando dados 2025

In [ ]:
modalidades = [5, 6] # Pegar todas as modalidades de licitação, mas para fins de teste, vamos pegar apenas as principais.
todos_registros = []

In [ ]:
for modalidade in modalidades:
    pagina = 1
    total_modalidade = 0

    while True:
        params = {
            "pagina": pagina,
            "tamanhoPagina": 500,
            "dataPublicacaoPncpInicial": "2024-01-01",
            "dataPublicacaoPncpFinal": "2024-12-31",
            "codigoModalidade": modalidade,
            "unidadeOrgaoUfSigla": "SP"
        }

        resposta = requests.get(url, params=params, timeout=60)

        if resposta.status_code == 429:
            espera = 5
            print(f"Modalidade {modalidade}, página {pagina}: 429, aguardando {espera}s e tentando de novo...")
            time.sleep(espera)
            continue  # tenta a mesma página de novo, sem avançar

        if resposta.status_code != 200:
            print(f"Modalidade {modalidade}, página {pagina}: erro {resposta.status_code}")
            break

        dados = resposta.json()
        registros = extrair_registros(dados)

        if not registros:
            break

        todos_registros.extend(registros)
        total_modalidade += len(registros)

        if len(registros) < 500:
            break

        pagina += 1
        time.sleep(0.2)  # pausa maior entre páginas

    print(f"Modalidade {modalidade}: {total_modalidade} registros coletados.")
    time.sleep(3)  # pausa entre modalidades, para o servidor "esfriar"

In [ ]:
# print("JSON retornado:", registros)
df_25 = pd.json_normalize(todos_registros)
df_25.head()

In [ ]:

# 2. PROCURA AUTOMÁTICA: Procura qualquer coluna que tenha 'srp' ou 'preco' no nome
colunas_srp = [
    c
    for c in df_25.columns
    if 'srp' in c.lower() or 'registropreco' in c.lower() or 'preco' in c.lower()
]
print("Colunas encontradas relacionadas a SRP/Preço:")
print(colunas_srp)

# Se encontrou alguma coluna compatível, renomeia a primeira para 'srp'
if colunas_srp:
    coluna_identificada = colunas_srp[0]
    print(f"\nUsando a coluna '{coluna_identificada}' como 'srp'")
    df_25 = df_25.rename(columns={coluna_identificada: 'srp'})
else:
    print(
        "\nNenhuma coluna de SRP foi encontrada diretamente. Veja todas as colunas:"
    )
    print(df_25.columns.tolist())

# 3. Mapeia outras colunas comuns do PNCP
mapeamento = {
    'orgaoEntidade.razaoSocial': 'orgaoEntidadeRazaoSocial',
    'modalidadeNome': 'modalidadeNome',
    'numeroCompra': 'numeroCompra',
    'objetoCompra': 'objetoCompra',
    'valorTotalEstimado': 'valorTotalEstimado',
    'valorTotalHomologado': 'valorTotalHomologado',
}
df_25 = df_25.rename(
    columns={k: v for k, v in mapeamento.items() if k in df_25.columns}
)

df_25["ano"] = 2025

# 4. Trata e padroniza a coluna SRP (identifica True, 1, 'True', 'S', etc.)
if 'srp' in df_25.columns:
    # Mostra os valores brutos que vieram da API antes de converter
    print("\nValores brutos encontrados na coluna SRP:")
    print(df_25['srp'].value_counts(dropna=False))

    # Converte para booleano real
    df_25['srp_bool'] = df_25['srp'].astype(str).str.lower().isin(['true', '1', 's', 'sim'])

    df_25_com_srp = df_25[df_25['srp_bool'] == True]
    df_25_sem_srp = df_25[df_25['srp_bool'] == False]

    print(f"\n Total COM SRP: {len(df_25_com_srp)}")
    print(f" Total SEM SRP: {len(df_25_sem_srp)}")


In [ ]:
print("\nDataFrame com SRP:")
display(df_25_com_srp.head())
print("\nDataFrame sem SRP:")
display(df_25_sem_srp.head())

In [ ]:
# GroupBy agregando com nunique (únicos) e count (total de registros)
df_qnt_modalidade_orgao_25 = df_25.groupby('srp').agg(
    qtd_modalidades=('modalidadeNome', 'nunique'),
    qtd_orgaos=('orgaoEntidadeRazaoSocial', 'nunique'),
    total_registros=('numeroCompra', 'count')
)

# Renomeia os índices para facilitar a leitura no relatório
df_qnt_modalidade_orgao_25.index = ['Sem SRP', 'Com SRP']
display(df_qnt_modalidade_orgao_25)

# Comparação 2024 X 2025

In [ ]:
display(df_qnt_modalidade_orgao_24)
display(df_qnt_modalidade_orgao_25)

In [ ]:
df_total = pd.concat([df_24, df_25], ignore_index=True)
df_total

In [ ]:
# 1. Lista com o nome exato das colunas de maior relevância
colunas_principais = [
    'srp',
    'ano',
    'numeroCompra',
    'modalidadeNome',
    'orgaoEntidadeRazaoSocial',
    'objetoCompra',
    'valorTotalEstimado',
    'valorTotalHomologado',
    'dataPublicacaoPncp',
]

# 2. Seleciona apenas as colunas que realmente existem no df_total
colunas_presentes = [c for c in colunas_principais if c in df_total.columns]
df_final_reduzido = df_total[colunas_presentes].copy()

# 3. Tratamento de Tipos de Dados


# B) Converte colunas financeiras para numérico (Float)
for col_valor in ['valorTotalEstimado', 'valorTotalHomologado']:
    if col_valor in df_final_reduzido.columns:
        df_final_reduzido[col_valor] = pd.to_numeric(
            df_final_reduzido[col_valor], errors='coerce'
        )

# C) Converte a data de publicação para formato de data real
if 'dataPublicacaoPncp' in df_final_reduzido.columns:
    df_final_reduzido['dataPublicacaoPncp'] = pd.to_datetime(
        df_final_reduzido['dataPublicacaoPncp'], errors='coerce'
    )

print("=== DATAFRAME FINAL REDUZIDO ===")
print(f"Dimensões do DataFrame: {df_final_reduzido.shape}")
display(df_final_reduzido.head(10))

In [ ]:
df_final_reduzido.to_parquet("srp_contratacoes_tratado.parquet", index=False)

In [ ]:
df_final_reduzido = pd.read_parquet("srp_contratacoes_tratado.parquet")